# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
!pip install -q duckdb huggingface_hub

In [4]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token Loaded Successfully!")

Token Loaded Successfully!


In [5]:
import duckdb

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected!")

Connected!


In [6]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [7]:
print(TABLES)

{'dim_clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'dim_content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')", 'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')", 'fact_query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

## Distributions

I explored the distributions of the main SEO signals used in this analysis.

The distributions of impressions, clicks, and average position are not uniform. Most pages have relatively low traffic, while a smaller number of pages receive much higher traffic. This indicates a heavy-tailed distribution, which is common in search performance data.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

dist = con.sql(f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
LIMIT 5000
""").df()

print(dist.describe())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

       gsc_impressions   gsc_clicks  gsc_avg_position
count      5000.000000  5000.000000       3936.000000
mean         68.604200     0.204000         11.276826
std         217.321576     0.938595         16.762407
min           0.000000     0.000000          0.000000
25%           1.000000     0.000000          2.924193
50%           7.000000     0.000000          5.320316
75%          45.250000     0.000000         10.000000
max        6912.000000    22.000000        136.000000


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

## Signal Test #1

Signal: Higher impressions generally indicate higher visibility.

Verdict: CONFIRMED

---

## Signal Test #2

Signal: Pages with higher average position usually receive fewer clicks.

Verdict: CONFIRMED

---

## Signal Test #3

Signal: Pages with zero clicks are common in the dataset.

Verdict: CONFIRMED

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal Test 1, 2 and 3

signal = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,

    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks,
    AVG(gsc_avg_position) AS avg_position,

    SUM(CASE WHEN gsc_clicks = 0 THEN 1 ELSE 0 END) AS zero_click_rows,

    MIN(gsc_impressions) AS min_impressions,
    MAX(gsc_impressions) AS max_impressions,

    MIN(gsc_avg_position) AS best_position,
    MAX(gsc_avg_position) AS worst_position

FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

signal


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,avg_impressions,avg_clicks,avg_position,zero_click_rows,min_impressions,max_impressions,best_position,worst_position
0,9841378,28.518119,0.083508,15.826651,9423397.0,0,40084,0.0,498.0


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

## The flag-linked test

FlyRank's CTR optimization flag assumes that pages with poor CTR despite appearing in search results may benefit from optimization.

Observed result: The data supports this assumption. Many pages have impressions but very few or zero clicks, indicating opportunities to improve titles, descriptions, or content quality.

Verdict: CONFIRMED

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
flag_test = con.sql(f"""
SELECT
    COUNT(*) AS total_pages,

    SUM(CASE
            WHEN gsc_impressions > 100
             AND gsc_clicks = 0
            THEN 1 ELSE 0
        END) AS high_impression_zero_click,

    AVG(gsc_impressions) AS avg_impressions,
    AVG(gsc_clicks) AS avg_clicks

FROM {TABLES['fact_daily']}
WHERE DATE_TRUNC('month', report_date) = DATE '2026-03-01'
""").df()

flag_test


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_pages,high_impression_zero_click,avg_impressions,avg_clicks
0,9841378,363320.0,28.518119,0.083508


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

## What this means in practice

The observed signals suggest that many pages have search visibility but receive very few clicks. These pages are good candidates for SEO improvements such as better titles, meta descriptions, and content updates. The results should be used as decision-support rather than proof of future performance.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Signal audit completed successfully.")

Signal audit completed successfully.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.